# TSP restringido mediante QUBO y Simulated Annealing

Problema de optimización inspirado en el viajante de comercio (TSP).

Objetivo: ruta de coste mínimo que:

- parte de la ciudad 0 y termina en la ciudad 0,
- visita una sola vez cada ciudad,
- impone que la ciudad **2** sea la **segunda ciudad visitada** tras salir del origen:
  $0 \rightarrow x \rightarrow 2 \rightarrow \cdots \rightarrow 0$.

El problema se modela como QUBO (formato compatible con annealing tipo D-Wave).
Sin acceso a solvers de D-Wave, se resuelve con **Simulated Annealing**, técnica clásica
relacionada conceptualmente con Quantum Annealing.


## Librerías


In [ ]:
import itertools
import random
import math

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx


## 1. Definición del grafo

Cinco ciudades: $V = \{0,1,2,3,4\}$. La ciudad $0$ es el origen.

Distancias del enunciado:

$d_{01}=3,\quad d_{03}=4,\quad d_{04}=2$

$d_{12}=3,\quad d_{13}=4$

$d_{23}=1,\quad d_{24}=6$

La distancia $0$–$2$ no viene en el enunciado; se asigna $d_{02}=4$
(misma escala, sin favorecer en exceso esa arista).


In [ ]:
nodes = [0, 1, 2, 3, 4]
origin = 0

# Distancia elegida para la arista no especificada en el enunciado
d02 = 4

dist = {
    (0, 1): 3,
    (0, 2): d02,
    (0, 3): 4,
    (0, 4): 2,
    (1, 2): 3,
    (1, 3): 4,
    (2, 3): 1,
    (2, 4): 6,
}

# Grafo no dirigido: distancias simétricas
for (i, j), value in list(dist.items()):
    dist[(j, i)] = value

# Penalización para aristas inexistentes
BIG_M = 100


def edge_exists(i, j):
    return (i, j) in dist


def cost(i, j):
    """Coste de viajar de i a j; arista inexistente → penalización grande."""
    if i == j:
        return BIG_M
    return dist.get((i, j), BIG_M)


## 2. Visualización del grafo

Las aristas inexistentes no se dibujan. En el modelo se penalizan con $M$.


In [ ]:
G = nx.Graph()
for node in nodes:
    G.add_node(node)

for (i, j), w in dist.items():
    if i < j:
        G.add_edge(i, j, weight=w)

pos = {
    0: (0, 0),
    1: (1, 1),
    2: (2, 0),
    3: (1, -1),
    4: (-1, -1),
}

plt.figure(figsize=(7, 5))
nx.draw(G, pos, with_labels=True, node_size=1200, font_size=16)
labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=12)
plt.title("Grafo del TSP restringido")
plt.show()


## 3. Restricción sobre la ciudad 2

La ciudad $2$ debe ser la **segunda ciudad visitada** después de salir del origen:

$0 \rightarrow x \rightarrow 2 \rightarrow \cdots \rightarrow 0$

Es decir, en la secuencia de la ruta, la ciudad $2$ aparece en la posición de índice $2$
(`route[2] == 2`).


In [ ]:
def route_cost(route):
    """Coste total de una ruta."""
    return sum(cost(route[i], route[i + 1]) for i in range(len(route) - 1))


def is_valid_route(route):
    """Comprueba inicio/fin en origen, visita única y restricción de la ciudad 2."""
    if route[0] != origin or route[-1] != origin:
        return False
    if sorted(route[1:-1]) != [1, 2, 3, 4]:
        return False
    # Segunda ciudad visitada tras el origen = 2
    return route[2] == 2


## 4. Generación de rutas candidatas

Con cuatro ciudades distintas del origen hay $4! = 24$ permutaciones antes de la restricción.
Se conservan solo las que cumplen `route[2] == 2` y se calcula su coste.

Aristas inexistentes → penalización $M=100$.


In [ ]:
def generate_candidate_routes():
    """Genera todas las rutas candidatas que cumplen las restricciones."""
    candidates = []
    for perm in itertools.permutations([1, 2, 3, 4]):
        route = [origin] + list(perm) + [origin]
        if is_valid_route(route):
            candidates.append({
                "route": route,
                "cost": route_cost(route),
                "uses_invalid_edges": any(
                    not edge_exists(route[i], route[i + 1])
                    for i in range(len(route) - 1)
                ),
            })
    return candidates


candidates = generate_candidate_routes()
print(f"Rutas candidatas: {len(candidates)}")
for candidate in candidates:
    print(
        candidate["route"],
        "coste =", candidate["cost"],
        "usa aristas inexistentes =", candidate["uses_invalid_edges"],
    )


## 5. Solución óptima de referencia

Antes del QUBO, se calcula la mejor ruta por enumeración (referencia para validar el annealing).

Se distingue entre:

- rutas solo con aristas existentes,
- rutas que usan alguna arista inexistente (penalización $M$).


In [ ]:
def best_candidate(candidates, allow_invalid_edges=True):
    pool = candidates
    if not allow_invalid_edges:
        pool = [c for c in candidates if not c["uses_invalid_edges"]]
    if not pool:
        return None
    return min(pool, key=lambda c: c["cost"])


best_with_penalty = best_candidate(candidates, allow_invalid_edges=True)
best_strict = best_candidate(candidates, allow_invalid_edges=False)

print("Mejor ruta permitiendo penalizaciones:")
print(best_with_penalty)
print("\nMejor ruta usando solo aristas existentes:")
print(best_strict)


## 6. Formulación QUBO reducida

Modelo QUBO reducido: una variable binaria por ruta candidata

$y_k =
\begin{cases}
1, & \text{si se selecciona la ruta } k \\
0, & \text{en caso contrario}
\end{cases}$

Energía:

$E(y)=\sum_k c_k y_k + A\left(1-\sum_k y_k\right)^2$

- $c_k$: coste de la ruta $k$,
- $A$: penalización por seleccionar cero o más de una ruta.

Es QUBO porque la función es cuadrática en variables binarias.


In [ ]:
def qubo_energy(state, candidates, penalty=100):
    """Energía QUBO: state[k]=1 selecciona la ruta candidata k."""
    costs = np.array([candidate["cost"] for candidate in candidates])
    y = np.array(state)
    route_cost_term = np.dot(costs, y)
    constraint_penalty = penalty * (1 - np.sum(y)) ** 2
    return route_cost_term + constraint_penalty


## 7. Ejemplo de energía QUBO

Si se selecciona exactamente una ruta, la penalización es cero y la energía coincide con su coste.
Si se seleccionan cero o varias rutas, aparece la penalización $A$.


In [ ]:
print("Rutas candidatas:")
for k, candidate in enumerate(candidates):
    print(k, candidate)

print("\nEjemplo seleccionando la primera ruta:")
state = [1] + [0] * (len(candidates) - 1)
print("Estado:", state)
print("Energía:", qubo_energy(state, candidates, penalty=100))

print("\nEjemplo seleccionando ninguna ruta:")
state = [0] * len(candidates)
print("Estado:", state)
print("Energía:", qubo_energy(state, candidates, penalty=100))

print("\nEjemplo seleccionando dos rutas:")
state = [1, 1] + [0] * (len(candidates) - 2)
print("Estado:", state)
print("Energía:", qubo_energy(state, candidates, penalty=100))


## 8. Resolución mediante Simulated Annealing

Simulated Annealing parte de un estado inicial y propone cambios aleatorios.
Si el cambio mejora la energía, se acepta; si empeora, puede aceptarse con

$P = e^{-\Delta E/T}$

Al inicio $T$ es alta (más exploración); al enfriar, el algoritmo se vuelve más selectivo.


In [ ]:
def random_state(n):
    """Estado binario aleatorio de longitud n."""
    return [random.randint(0, 1) for _ in range(n)]


def neighbor_state(state):
    """Vecino: flip de un bit aleatorio."""
    new_state = state.copy()
    index = random.randint(0, len(state) - 1)
    new_state[index] = 1 - new_state[index]
    return new_state


def simulated_annealing(
    candidates,
    penalty=100,
    max_iter=5000,
    initial_temperature=50,
    cooling_rate=0.995,
    seed=42,
):
    """Resuelve el QUBO reducido mediante Simulated Annealing."""
    random.seed(seed)

    n = len(candidates)
    current_state = random_state(n)
    current_energy = qubo_energy(current_state, candidates, penalty)

    best_state = current_state.copy()
    best_energy = current_energy
    temperature = initial_temperature
    history = []

    for _ in range(max_iter):
        candidate_state = neighbor_state(current_state)
        candidate_energy = qubo_energy(candidate_state, candidates, penalty)
        delta = candidate_energy - current_energy

        if delta < 0:
            accept = True
        else:
            accept_probability = math.exp(-delta / temperature) if temperature > 0 else 0
            accept = random.random() < accept_probability

        if accept:
            current_state = candidate_state
            current_energy = candidate_energy

        if current_energy < best_energy:
            best_state = current_state.copy()
            best_energy = current_energy

        history.append(current_energy)
        temperature *= cooling_rate

    return best_state, best_energy, history


## 9. Ejecución del algoritmo

Se ejecuta Simulated Annealing y se decodifica la ruta seleccionada por el estado binario óptimo.


In [ ]:
def decode_selected_route(state, candidates):
    """Convierte un estado binario en la ruta seleccionada."""
    selected_indices = [i for i, bit in enumerate(state) if bit == 1]
    if len(selected_indices) != 1:
        return None
    return candidates[selected_indices[0]]


best_state, best_energy, history = simulated_annealing(
    candidates,
    penalty=100,
    max_iter=5000,
    initial_temperature=50,
    cooling_rate=0.995,
    seed=42,
)

selected_route = decode_selected_route(best_state, candidates)

print("Mejor estado binario:", best_state)
print("Mejor energía:", best_energy)
print("Ruta seleccionada:", selected_route)
print("Óptimo por enumeración:", best_strict)


## 10. Análisis de resultados

Simulated Annealing obtiene la ruta

$[0, 4, 2, 3, 1, 0]$

con coste $16$. Usa solo aristas existentes y coincide con el óptimo por enumeración.


## 11. Convergencia del algoritmo

La energía baja hasta el mínimo global y se estabiliza: comportamiento típico de Simulated Annealing
y coherente con el QUBO reducido.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

axs[0].plot(history, color="#0E4A75", lw=1)
axs[0].set_xlabel("Iteraciones")
axs[0].set_ylabel("Energía actual")
axs[0].set_title("Convergencia de Simulated Annealing")
axs[0].grid(True, linestyle="--", alpha=0.5)

window = 50
smoothed = np.convolve(history, np.ones(window) / window, mode="valid")
axs[1].plot(smoothed, color="#219BC3", lw=1.5)
axs[1].set_xlabel("Iteraciones")
axs[1].set_ylabel(f"Energía (media móvil {window})")
axs[1].set_title("Convergencia suavizada")
axs[1].grid(True, linestyle="--", alpha=0.5)

fig.tight_layout()
plt.show()
